# Datenaufbereitung: World Happiness Report

Liest den World Happiness Report eines Jahres (`data/worldhappinessreport/<jahr>.csv`) ein, vereinheitlicht die Spaltennamen der verschiedenen Jahrgänge und speichert die Werte in der Tabelle `metric`.

Voraussetzung: Datenbank gemäss `db/schema.sql` und gesetzte Umgebungsvariablen (siehe `.env.example`).

In [ ]:
import pandas as pd
from mariadb import DataError

from bina_db import connect, CountryLookup, get_or_create_metric_definition

In [ ]:
# Jahr des World Happiness Reports, der importiert werden soll
year = 2023

# Die Dateien verwenden je nach Jahr Komma, Semikolon oder Tab als Trennzeichen
whr = pd.read_csv(f"../data/worldhappinessreport/{year}.csv", sep=None, engine="python", encoding="utf-8-sig")

In [ ]:
maria_db_connection = connect()
country_lookup = CountryLookup(maria_db_connection)

In [104]:
whr

,Country name,Ladder score,Standard error of ladder score,upperwhisker,lowerwhisker,Logged GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Ladder score in Dystopia,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,Finland,7.804,0.036,7.875,7.733,10.792,0.969,71.150,0.961,-0.019,0.182,1.778,1.888,1.585,0.535,0.772,0.126,0.535,2.363
1,Denmark,7.586,0.041,7.667,7.506,10.962,0.954,71.250,0.934,0.134,0.196,1.778,1.949,1.548,0.537,0.734,0.208,0.525,2.084
2,Iceland,7.530,0.049,7.625,7.434,10.896,0.983,72.050,0.936,0.211,0.668,1.778,1.926,1.620,0.559,0.738,0.250,0.187,2.250
3,Israel,7.473,0.032,7.535,7.411,10.639,0.943,72.697,0.809,-0.023,0.708,1.778,1.833,1.521,0.577,0.569,0.124,0.158,2.691
4,Netherlands,7.403,0.029,7.460,7.346,10.942,0.930,71.550,0.887,0.213,0.379,1.778,1.942,1.488,0.545,0.672,0.251,0.394,2.110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,Congo (Kinshasa),3.207,0.095,3.394,3.020,7.007,0.652,55.375,0.664,0.086,0.834,1.778,0.531,0.784,0.105,0.375,0.183,0.068,1.162
133,Zimbabwe,3.204,0.061,3.323,3.084,7.641,0.690,54.050,0.654,-0.046,0.766,1.778,0.758,0.881,0.069,0.363,0.112,0.117,0.905
134,Sierra Leone,3.138,0.082,3.299,2.976,7.394,0.555,54.900,0.660,0.105,0.858,1.778,0.670,0.540,0.092,0.371,0.193,0.051,1.221
135,Lebanon,2.392,0.044,2.479,2.305,9.478,0.530,66.149,0.474,-0.141,0.891,1.778,1.417,0.476,0.398,0.123,0.061,0.027,-0.110


In [ ]:
# Die Spaltennamen ändern sich von Jahr zu Jahr -> einheitliche Metriknamen
def map_metric_name(metric_name):
    if metric_name in ["Happiness Rank", "Happiness.Rank", "Overall rank", "RANK"]:
        return "Happiness Rank"
    elif metric_name in ["Happiness Score", "Happiness.Score", "Score"]:
        return "Happiness Score"
    elif metric_name in ["Standard Error", "Standard.Error"]:
        return "Standard Error"
    elif metric_name in ["Economy (GDP per Capita)", "Economy..GDP.per.Capita.", "GDP per capita", "Explained by: Log GDP per capita"]:
        return "Economy (GDP per Capita)"
    elif metric_name in ["Health (Life Expectancy)", "Health..Life.Expectancy.", "Healthy life expectancy", "Explained by: Healthy life expectancy"]:
        return "Health (Life Expectancy)"
    elif metric_name in ["Trust (Government Corruption)", "Trust..Government.Corruption.", "Perceptions of corruption", "Explained by: Perceptions of corruption"]:
        return "Trust (Government Corruption)"
    elif metric_name in ["Freedom", "Freedom to make life choices", "Explained by: Freedom to make life choices"]:
        return "Freedom"
    elif metric_name in ["Dystopia Residual", "Dystopia.Residual", "Dystopia + residual", "Dystopia (1.83) + residual"]:
        return "Dystopia Residual"
    elif metric_name in ["Generosity", "Explained by: Generosity"]:
        return "Generosity"
    elif metric_name in ["Social support", "Explained by: Social support"]:
        return "Social support"
    else:
        return metric_name

In [ ]:
country_column_names = ["Country name", "Country", "Country or region"]
country_column_name = next(name for name in country_column_names if name in whr.columns)

metrics = ["Happiness Rank", "Happiness.Rank", "Overall rank", "RANK", "Happiness Score", "Happiness.Score", "Score", "Standard Error", "Standard.Error", "Economy (GDP per Capita)", "Economy..GDP.per.Capita.", "GDP per capita", "Explained by: Log GDP per capita", "Family", "Health (Life Expectancy)", "Health..Life.Expectancy.", "Healthy life expectancy", "Explained by: Healthy life expectancy", "Freedom", "Freedom to make life choices", "Explained by: Freedom to make life choices", "Trust (Government Corruption)", "Trust..Government.Corruption.", "Perceptions of corruption", "Generosity", "Explained by: Generosity", "Dystopia Residual", "Dystopia.Residual", "Dystopia + residual", "Dystopia (1.83) + residual", "Social support", "Explained by: Social support"]

insert_statement = "insert into metric(value, year, country, metric_definition) VALUES (?, ?, ?, ?)"
cursor = maria_db_connection.cursor()

# Nicht jede Spaltenvariante existiert in jedem Jahr -> nur vorhandene Spalten importieren
for metric in [m for m in metrics if m in whr.columns]:
    metric_definition_id = get_or_create_metric_definition(maria_db_connection, map_metric_name(metric))

    for index, row in whr[[country_column_name, metric]].iterrows():
        country_name = row[country_column_name]
        value = row[metric]

        country_id = country_lookup.lookup(country_name)
        if country_id is None:
            print("could not find country id: " + country_name)
            continue

        try:
            cursor.execute(insert_statement, (value, year, country_id, metric_definition_id))
        except DataError:
            print("could not insert value: " + str(value) + " for metric: " + str(metric))

maria_db_connection.commit()
maria_db_connection.close()